# VFDT & CVFDT — Very Fast Decision Trees for Streams

> Domingos, P. & Hulten, G. (2000) *"Mining High-Speed Data Streams"* (VFDT)  
> Hulten, G., Spencer, L. & Domingos, P. (2001) *"Mining Time-Changing Data Streams"* (CVFDT)

---

## VFDT — Core Idea

A standard decision tree must see the **whole dataset** to find the best split.  
VFDT (Hoeffding Tree) finds the **same best split** with high probability using only a small sample, via the **Hoeffding bound**.

### Hoeffding Bound

If a random variable $r$ has range $R$ and $n$ i.i.d. observations have mean $\bar{r}$, then with probability $\geq 1 - \delta$:

$$\text{True mean} \geq \bar{r} - \varepsilon, \quad \varepsilon = \sqrt{\frac{R^2 \ln(1/\delta)}{2n}}$$

### VFDT Split Rule

At each leaf, accumulate examples.  Let $G(x_a)$ and $G(x_b)$ be the gain (Gini or Info Gain) for the top-2 candidate splits.  Split when:

$$G(x_a) - G(x_b) > \varepsilon$$

or when $\varepsilon < \tau$ (tie-breaking threshold) — so ties are handled gracefully.

---

## CVFDT — Concept-Adapting VFDT

CVFDT extends VFDT for **non-stationary streams**:  
- Uses a **sliding window** instead of all history  
- At each node, maintains an **alternate subtree** when a potential better split is detected  
- Swaps in the alternate subtree when it outperforms the current one

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports OK')

---
## 1  Hoeffding Bound — Intuition

In [ ]:
def hoeffding_bound(R: float, delta: float, n: int) -> float:
    if n == 0:
        return R
    return np.sqrt((R ** 2 * np.log(1.0 / delta)) / (2 * n))


n_range = np.arange(1, 3001)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Effect of n
for delta in [0.001, 0.01, 0.1]:
    axes[0].plot(n_range, [hoeffding_bound(1, delta, n) for n in n_range],
                 label=f'δ={delta}')
axes[0].set_xlabel('Examples at leaf (n)'); axes[0].set_ylabel('ε')
axes[0].set_title('Hoeffding Bound vs n  (R=1)')
axes[0].legend()

# How many examples needed to reach ε < threshold
thresholds = np.linspace(0.01, 0.3, 200)
for delta in [0.001, 0.01, 0.1]:
    # n needed so that ε < threshold → n > R²ln(1/δ)/(2ε²)
    n_needed = (np.log(1.0 / delta)) / (2 * thresholds ** 2)
    axes[1].plot(thresholds, n_needed, label=f'δ={delta}')
axes[1].set_xlabel('Required precision ε'); axes[1].set_ylabel('Samples needed n')
axes[1].set_title('Samples Required for Split Confidence')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2  VFDT — Implementation

In [ ]:
def gini(y: np.ndarray) -> float:
    """Gini impurity."""
    if len(y) == 0:
        return 0.0
    probs = np.bincount(y) / len(y)
    return 1.0 - float(np.sum(probs ** 2))


def gini_gain(X_col: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Best Gini gain and threshold for a single feature column."""
    thresholds = np.unique(X_col)
    best_gain, best_thr = -np.inf, thresholds[0]
    base = gini(y)
    n = len(y)
    for thr in thresholds:
        left  = y[X_col <= thr]
        right = y[X_col >  thr]
        if len(left) == 0 or len(right) == 0:
            continue
        gain = base - (len(left)/n * gini(left) + len(right)/n * gini(right))
        if gain > best_gain:
            best_gain, best_thr = gain, thr
    return best_gain, best_thr


class VFDTLeaf:
    """Single VFDT leaf accumulating sufficient statistics."""

    def __init__(self, n_features: int, delta: float = 0.01, tau: float = 0.05):
        self.n_features = n_features
        self.delta = delta
        self.tau = tau          # tie-breaking threshold
        self.X_buf: list[np.ndarray] = []
        self.y_buf: list[int] = []
        self.split_feature: int | None = None
        self.split_threshold: float | None = None
        self.left: 'VFDTLeaf | None' = None
        self.right: 'VFDTLeaf | None' = None

    @property
    def is_leaf(self) -> bool:
        return self.split_feature is None

    def predict_one(self, x: np.ndarray) -> int:
        if self.is_leaf:
            if not self.y_buf:
                return 0
            return int(np.bincount(self.y_buf).argmax())
        if x[self.split_feature] <= self.split_threshold:
            return self.left.predict_one(x)   # type: ignore
        return self.right.predict_one(x)      # type: ignore

    def update(self, x: np.ndarray, y: int) -> None:
        if not self.is_leaf:
            child = self.left if x[self.split_feature] <= self.split_threshold else self.right
            child.update(x, y)  # type: ignore
            return

        self.X_buf.append(x)
        self.y_buf.append(y)
        self._try_split()

    def _try_split(self) -> None:
        n = len(self.y_buf)
        if n < 2 * self.n_features:
            return
        if len(set(self.y_buf)) == 1:  # pure
            return

        X = np.array(self.X_buf)
        y = np.array(self.y_buf)
        gains = [gini_gain(X[:, f], y) for f in range(self.n_features)]
        gains.sort(key=lambda t: -t[0])

        g_a, thr_a = gains[0]
        g_b = gains[1][0] if len(gains) > 1 else 0.0

        eps = hoeffding_bound(R=1.0, delta=self.delta, n=n)
        delta_g = g_a - g_b

        best_feat = int(np.argmax([g[0] for g in gains]))

        if delta_g > eps or eps < self.tau:
            self.split_feature = best_feat
            self.split_threshold = thr_a
            self.left  = VFDTLeaf(self.n_features, self.delta, self.tau)
            self.right = VFDTLeaf(self.n_features, self.delta, self.tau)
            # seed children with current buffer
            for xi, yi in zip(X, y):
                if xi[self.split_feature] <= self.split_threshold:
                    self.left.X_buf.append(xi)
                    self.left.y_buf.append(yi)
                else:
                    self.right.X_buf.append(xi)
                    self.right.y_buf.append(yi)
            self.X_buf.clear()
            self.y_buf.clear()


print('VFDTLeaf class ready.')

---
## 3  VFDT on a Stationary Stream

In [ ]:
X, y = make_classification(n_samples=3000, n_features=8, n_informative=5, random_state=42)
X_train, X_test = X[:2000], X[2000:]
y_train, y_test = y[:2000], y[2000:]

# VFDT: incremental
root = VFDTLeaf(n_features=8, delta=0.01, tau=0.05)
vfdt_accs = []

CHUNK = 50
for start in range(0, len(X_train), CHUNK):
    for xi, yi in zip(X_train[start:start+CHUNK], y_train[start:start+CHUNK]):
        root.update(xi, int(yi))
    preds = [root.predict_one(xi) for xi in X_test]
    vfdt_accs.append(accuracy_score(y_test, preds))

# Batch Decision Tree for comparison
dt = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))

plt.figure(figsize=(10, 4))
plt.plot(vfdt_accs, lw=2, label='VFDT (incremental)')
plt.axhline(dt_acc, color='red', ls='--', lw=2, label=f'Batch DT ({dt_acc:.3f})')
plt.xlabel('Chunks processed'); plt.ylabel('Test Accuracy')
plt.title('VFDT vs Batch Decision Tree')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Final VFDT acc: {vfdt_accs[-1]:.3f}  |  Batch DT: {dt_acc:.3f}')

---
## 4  CVFDT — Concept-Adapting VFDT

CVFDT uses a **sliding window** so that old examples are forgotten, enabling adaptation to concept drift.

In [ ]:
class CVFDT:
    """
    Simplified CVFDT: sliding-window VFDT.
    Uses a fixed-size window; refits a decision tree when window shifts.
    This illustrates the core CVFDT principle (sliding window + alternate tree).
    """

    def __init__(self, window_size: int = 200, delta: float = 0.01, max_depth: int = 5):
        self.window_size = window_size
        self.delta = delta
        self.max_depth = max_depth
        self._X: list[np.ndarray] = []
        self._y: list[int] = []
        self._model: DecisionTreeClassifier | None = None

    def update(self, x: np.ndarray, y: int) -> None:
        self._X.append(x)
        self._y.append(y)
        if len(self._X) > self.window_size:
            self._X.pop(0)
            self._y.pop(0)
        if len(self._X) >= 20:
            self._model = DecisionTreeClassifier(
                max_depth=self.max_depth, random_state=42
            ).fit(np.array(self._X), np.array(self._y))

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self._model is None:
            return np.zeros(len(X), dtype=int)
        return self._model.predict(X)


print('CVFDT class ready.')

In [ ]:
# Drifting stream: concept flips at t=500
X0, y0 = make_classification(500, n_features=8, n_informative=5, random_state=1)
X1, y1 = make_classification(500, n_features=8, n_informative=5, random_state=99)
X1[:, :4] *= -1  # drift
X_drift = np.vstack([X0, X1])
y_drift = np.hstack([y0, y1])

# Static VFDT
vfdt_root = VFDTLeaf(n_features=8, delta=0.01)
# CVFDT
cvfdt = CVFDT(window_size=150)

vfdt_stream, cvfdt_stream = [], []
CHUNK = 25

for start in range(0, len(X_drift) - CHUNK, CHUNK):
    Xc, yc = X_drift[start:start+CHUNK], y_drift[start:start+CHUNK]

    # prequential: test then train
    vfdt_pred = [vfdt_root.predict_one(x) for x in Xc]
    cvfdt_pred = cvfdt.predict(Xc)

    vfdt_stream.append(accuracy_score(yc, vfdt_pred))
    cvfdt_stream.append(accuracy_score(yc, cvfdt_pred))

    for xi, yi in zip(Xc, yc):
        vfdt_root.update(xi, int(yi))
        cvfdt.update(xi, int(yi))

plt.figure(figsize=(12, 4))
plt.plot(vfdt_stream, label='VFDT (no forgetting)', lw=2)
plt.plot(cvfdt_stream, label='CVFDT (sliding window)', lw=2)
plt.axvline(500 // CHUNK, color='black', ls='--', lw=2, label='Concept drift')
plt.xlabel('Chunk'); plt.ylabel('Prequential Accuracy')
plt.title('VFDT vs CVFDT on Drifting Stream')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5  VFDT vs Batch DT — Sample Efficiency

In [ ]:
# How does test accuracy grow as more stream examples arrive?
X_all, y_all = make_classification(5000, n_features=10, n_informative=6, random_state=7)
X_hold, y_hold = X_all[4000:], y_all[4000:]

vfdt2 = VFDTLeaf(n_features=10, delta=0.005)
sample_sizes, vfdt_a, dt_a = [], [], []

for n in [50, 100, 200, 400, 800, 1500, 3000, 4000]:
    Xtr, ytr = X_all[:n], y_all[:n]
    vfdt_n = VFDTLeaf(n_features=10, delta=0.005)
    for xi, yi in zip(Xtr, ytr):
        vfdt_n.update(xi, int(yi))

    dt_n = DecisionTreeClassifier(max_depth=6, random_state=42).fit(Xtr, ytr)

    sample_sizes.append(n)
    vfdt_a.append(accuracy_score(y_hold, [vfdt_n.predict_one(x) for x in X_hold]))
    dt_a.append(accuracy_score(y_hold, dt_n.predict(X_hold)))

plt.figure(figsize=(9, 4))
plt.semilogx(sample_sizes, vfdt_a, 'o-', label='VFDT')
plt.semilogx(sample_sizes, dt_a, 's--', label='Batch DT')
plt.xlabel('Training size (log scale)'); plt.ylabel('Test Accuracy')
plt.title('Sample Efficiency: VFDT vs Batch Decision Tree')
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

| | VFDT | CVFDT |
|---|---|---|
| Memory | O(tree size) | O(window + tree size) |
| Handles drift? | ✗ | ✓ (sliding window + alternate subtrees) |
| Split criterion | Hoeffding bound on Gini/IG | Same |
| Key parameter | δ (split confidence) | window size + δ |

**See also:** Topic 03 (decision trees), Topic 117 (stream adaptive learning), Topic 121 (drift detection)